# 🔧 Proje #18: Driver Bulucu ve Kontrol Edici — Kapsamlı Teknik Rehber

Bu notebook dosyasında, Windows işletim sistemindeki donanım sürücülerini (driver) WMI altyapısı üzerinden sorgulayan, durumlarını tablo halinde gösteren ve kullanıcıya etkileşimli arama/filtreleme imkânı sunan masaüstü uygulamamızın çalışma mantığını, mimari yapısını ve kullandığı tüm teknolojileri derinlemesine inceleyeceğiz.

## 1. Windows Sürücü (Driver) Kavramı

### 1.1. Driver Nedir?

**Driver (Sürücü)**, işletim sistemi çekirdeği (kernel) ile donanım arasındaki iletişimi sağlayan yazılım katmanıdır. Her donanım parçası (ekran kartı, ses kartı, ağ adaptörü, USB denetleyici, klavye vb.) kendi sürücüsüne ihtiyaç duyar.

Sürücüler olmadan işletim sistemi donanıma nasıl komut göndereceğini bilemez. Örneğin:
- Ekran kartı sürücüsü olmadan monitörünüz düşük çözünürlükte takılı kalır.
- Ağ adaptörü sürücüsü olmadan Wi-Fi veya Ethernet çalışmaz.
- Ses kartı sürücüsü olmadan bilgisayarınız ses çıkarmaz.

### 1.2. Windows Sürücü Mimarisi

Windows'ta sürücüler **Kullanıcı Modu (User Mode)** ve **Çekirdek Modu (Kernel Mode)** olmak üzere iki katmanda çalışır:

```
┌───────────────────────────────────────────┐
│           Kullanıcı Uygulamaları          │ ← Bizim Python programımız burada
├───────────────────────────────────────────┤
│   WMI (Windows Management Instrumen.)    │ ← COM üzerinden donanım bilgilerini sunar
├───────────────────────────────────────────┤
│          Windows API (Win32 API)          │
├───────────────────────────────────────────┤
│     Kullanıcı Modu Sürücüleri (UMDF)     │
├═══════════════════════════════════════════┤
│     Çekirdek Modu Sürücüleri (KMDF)      │ ← Donanımla doğrudan iletişim kurar
├───────────────────────────────────────────┤
│       HAL (Hardware Abstraction Layer)    │
├───────────────────────────────────────────┤
│              DONANIM (Hardware)           │
└───────────────────────────────────────────┘
```

### 1.3. Plug and Play (PnP) Sistemi

Modern donanımlar **Plug and Play (PnP)** standardını destekler. Bu standart sayesinde:
1. Donanım bilgisayara takıldığında kendini otomatik olarak tanıtır.
2. İşletim sistemi, Device ID'yi kullanarak uygun sürücüyü arar ve yükler.
3. Kullanıcı müdahalesi gerekmeden donanım kullanıma hazır hale gelir.

Projemizde `Win32_PnPEntity` sınıfı tam olarak bu PnP cihazlarını sorgular.

## 2. WMI (Windows Management Instrumentation) Nedir?

WMI, Microsoft'un **işletim sistemi, donanım, yazılım, servisler ve ağ bileşenleri** hakkında yapılandırılmış veri sağlayan merkezi yönetim altyapısıdır.

### 2.1. WMI Mimarisi

```
┌──────────────────┐     ┌──────────────────┐
│  Python (wmi)    │────▶│   COM Arayüzü    │
│  Konsol/Script   │     │   (pywin32)      │
└──────────────────┘     └────────┬─────────┘
                                  │
                                  ▼
                     ┌──────────────────────┐
                     │  WMI Servisi         │
                     │  (winmgmt.exe)       │
                     └────────┬─────────────┘
                              │
                              ▼
              ┌──────────────────────────────┐
              │  CIM Repository (CIMOM)      │
              │  Sınıflar: Win32_PnPEntity,  │
              │  Win32_Processor,             │
              │  Win32_DiskDrive, ...         │
              └──────────────────────────────┘
```

### 2.2. WQL (WMI Query Language)

WMI'a sorgular **WQL** (SQL benzeri bir dil) ile yapılır. Python'daki `wmi` kütüphanesi bu sorguları otomatik oluşturur:

```python
# Python kodu:
computer.Win32_PnPEntity()

# Arka planda çalışan WQL sorgusu:
# SELECT * FROM Win32_PnPEntity
```

### 2.3. Win32_PnPEntity Sınıfının Döndürdüğü Özellikler

| Özellik | Açıklama | Örnek |
|---------|----------|-------|
| `Name` | Cihazın kullanıcı dostu adı | "Intel(R) UHD Graphics 730" |
| `DeviceID` | Cihazın benzersiz donanım tanımlayıcısı | "PCI\\VEN_8086&DEV_4692..." |
| `Manufacturer` | Üretici firma adı | "Intel Corporation" |
| `Status` | Sürücü durumu (OK, Error, Degraded vb.) | "OK" |
| `PNPDeviceID` | PnP alt sistemi tanımlayıcısı | DeviceID ile genellikle aynıdır |
| `ClassGuid` | Aygıt sınıfının GUID'i | "{4d36e972-e325-11ce-bfc1-08002be10318}" |
| `ConfigManagerErrorCode` | Hata kodu (0 = sorun yok) | 0 |

## 3. Uygulamanın Çalışma Akışı (Flow)

Uygulamanın baştan sona çalışma sırası şu şekildedir:

```
Kullanıcı programı başlatır
         │
         ▼
display_drivers_in_table() çağrılır
         │
         ▼
list_drivers() → WMI bağlantısı kurulur
         │
         ▼
Win32_PnPEntity sorgusu çalışır
         │
         ▼
Her cihaz için dict oluşturulur → drivers[] listesine eklenir
         │
         ▼
Tkinter penceresi oluşturulur
         │
         ├── Arama çubuğu eklenir
         ├── Treeview tablosu oluşturulur
         ├── populate_treeview(drivers) çağrılır
         │       │
         │       ├── Her satır eklenir
         │       └── Duruma göre tag atanır (yeşil/kırmızı)
         │
         ├── Olay bağlamaları yapılır:
         │       ├── <KeyRelease> → anlık arama
         │       └── <Double-1>  → Google'da arama
         │
         └── mainloop() ile olay döngüsü başlar
```

## 4. Kodun Bölüm Bölüm İncelenmesi

### 4.1. WMI Bağlantısı ve COM Arayüzü

```python
import wmi
computer = wmi.WMI()
```

Bu iki satır, arka planda şu işlemleri gerçekleştirir:

1. **`pywin32`** kütüphanesi üzerinden **COM (Component Object Model)** bağlantısı başlatılır.
2. COM arayüzü, `winmgmt.exe` servisine bağlanır.
3. WMI Repository'deki (CIM veritabanı) sınıflara erişim sağlanır.

**COM Nedir?**
COM, Microsoft'un farklı programlama dilleri ve bileşenler arasında iletişim kurmayı sağlayan ikili (binary) arayüz standardıdır. Python'dan Windows API'lerine erişmek için COM köprü vazifesi görür.

### 4.2. Sürücü Durumu Kontrolü

```python
status = "Yüklü" if device.Status == "OK" else "Yüklü Değil"
```

**`Status`** özelliğinin alabileceği değerler:

| Değer | Anlamı |
|-------|--------|
| `"OK"` | Cihaz düzgün çalışıyor |
| `"Error"` | Cihazda bir sorun var |
| `"Degraded"` | Cihaz çalışıyor ama performansı düşük |
| `"Unknown"` | Durum belirlenemiyor |
| `"Pred Fail"` | Cihaz yakında arızalanabilir |

### 4.3. JSON ile Veri Kalıcılığı (Persistence)

```python
json.dump(drivers, file, ensure_ascii=False, indent=4)
```

- **`ensure_ascii=False`**: Türkçe karakterlerin (`ü`, `ö`, `ş`, `ç`, `ğ`, `ı`) doğru kaydedilmesini sağlar. `True` olsaydı `"Yüklü"` yerine `"Y\u00fckl\u00fc"` yazılırdı.
- **`indent=4`**: JSON çıktısını okunabilir formatta girintili (pretty-print) yazar.

## 5. Tkinter Arayüzü — Detaylı İnceleme

### 5.1. ttk.Treeview Widget'ı

Treeview, Tkinter'daki en güçlü veri görselleştirme widget'larından biridir. Hem ağaç (tree) hem de tablo (grid) görünümünü destekler.

```python
columns = ("Cihaz", "Device ID", "Üretici", "Durum", "Sürücü Linki")
tree = ttk.Treeview(root, columns=columns, show="headings")
```

**`show="headings"` Parametresi:**
Varsayılan olarak Treeview, satırların solunda genişletilebilir bir ağaç ikonu (#0 sütunu) gösterir. `show="headings"` bunu gizleyerek saf tablo görünümü oluşturur.

### 5.2. Koşullu Renklendirme (Conditional Formatting)

Treeview'da satır renklerini değiştirmek için **tag** sistemi kullanılır:

```python
# 1. Adım: Etiket stilini tanımla
tree.tag_configure("yuklu", background="lightgreen")
tree.tag_configure("yuklu_degil", background="lightcoral")

# 2. Adım: Satır eklerken etiket ata
tree.item(item_id, tags=("yuklu",))
```

Bu yaklaşım, Excel'deki koşullu biçimlendirmeye (Conditional Formatting) benzer: Veri değerine göre satırın arka plan rengi dinamik olarak değişir.

### 5.3. Olay Bağlama (Event Binding)

Tkinter'da olay bağlama, belirli kullanıcı aksiyonlarını fonksiyonlara bağlamayı sağlar:

| Olay | Açıklama | Kullanım Amacı |
|------|----------|---------|
| `<KeyRelease>` | Tuşa basıp bırakıldığında tetiklenir | Anlık arama filtreleme |
| `<Double-1>` | Sol fare tuşuyla çift tıklama | Google'da sürücü arama |
| `<Button-1>` | Sol fare tuşuyla tek tıklama | Satır seçme |
| `<Return>` | Enter tuşuna basılması | Form gönderme |

### 5.4. webbrowser Modülü

```python
webbrowser.open(f"https://www.google.com/search?q={device_name} driver")
```

Bu modül, Python'ın standart kütüphanesinde yer alır ve işletim sisteminin varsayılan web tarayıcısını kullanarak URL açar. Platform bağımsızdır (Windows, macOS, Linux).

## 6. Arama ve Filtreleme Algoritması

Arama fonksiyonu, **liste kavramı (list comprehension)** ve **çoklu koşul** kullanarak verimli bir filtreleme yapar:

```python
filtered_drivers = [
    driver for driver in drivers
    if (driver.get("Cihaz") and query in driver["Cihaz"].lower()) or
       (driver.get("Device ID") and query in driver["Device ID"].lower()) or
       (driver.get("Üretici") and query in driver["Üretici"].lower()) or
       (driver.get("Durum") and query in driver["Durum"].lower())
]
```

**Algoritma Adımları:**
1. Kullanıcının girdiği metni küçük harfe çevir (`query`)
2. Tüm sürücüleri döngüyle tara
3. Her sürücünün 4 alanında (Cihaz, Device ID, Üretici, Durum) aranan metni kontrol et
4. Herhangi bir alanda eşleşme varsa o sürücüyü sonuç listesine dahil et
5. Filtrelenmiş listeyle tabloyu yeniden doldur

**Neden `driver.get()` Kullanılıyor?**
`.get()` metodu, anahtar bulunamazsa `None` döner ve `KeyError` hatası yerine güvenli bir kontrol sağlar. Bu, eksik veya bozuk veri gelmesi durumunda uygulamanın çökmesini önler.

**Performans Notu:**
Her tuş bırakma olayında (`<KeyRelease>`) tüm sürücüler yeniden taranır ve Treeview tamamen yeniden doldurulur. Tipik bir bilgisayarda ~100-300 aygıt olduğu düşünüldüğünde bu $O(n)$ karmaşıklığındaki işlem anlık sürer.

## 7. Device ID Anatomisi

Device ID, donanımı benzersiz şekilde tanımlayan bir dizedir. Yapısı şöyledir:

```
PCI\VEN_8086&DEV_A382&SUBSYS_7D481462&REV_10\3&11583659&0&B8
│   │        │        │                  │    │
│   │        │        │                  │    └── Instance Path (Benzersiz örnek yolu)
│   │        │        │                  └── Revision (Donanım revizyonu)
│   │        │        └── Subsystem ID (Alt sistem tanımlayıcı - Anakart üreticisi)
│   │        └── Device ID (Cihaz modeli)
│   └── Vendor ID (Üretici kodu - 8086 = Intel)
└── Bus Tipi (PCI, USB, ACPI, HID vb.)
```

**Yaygın Vendor ID'leri:**

| Vendor ID | Üretici |
|-----------|----------|
| `8086` | Intel |
| `10DE` | NVIDIA |
| `1002` | AMD/ATI |
| `10EC` | Realtek |
| `8087` | Intel (Bluetooth/Wireless) |
| `1B21` | ASMedia |

## 8. Uygulamalı Gösterim (Jupyter Üzerinde WMI Sorgusu)

Aşağıdaki kod bloğunda, WMI sorgusunun Jupyter Notebook ortamında doğrudan nasıl çalıştırıldığını ve ilk 5 cihazın bilgilerini görebilirsiniz.

In [ ]:
import wmi

try:
    computer = wmi.WMI()
    print("WMI bağlantısı başarıyla kuruldu!\n")
    
    # İlk 5 PnP cihazının bilgilerini yazdırma
    devices = list(computer.Win32_PnPEntity())
    print(f"Toplam tespit edilen cihaz sayısı: {len(devices)}\n")
    print("="*80)
    
    for i, device in enumerate(devices[:5]):
        print(f"\n--- Cihaz #{i+1} ---")
        print(f"  Ad       : {device.Name}")
        print(f"  Device ID: {device.DeviceID}")
        print(f"  Üretici  : {device.Manufacturer}")
        print(f"  Durum    : {device.Status}")
        print(f"  Sınıf    : {device.PNPClass}")
        
except Exception as e:
    print(f"Bu notebook yalnızca Windows üzerinde çalışır. Hata: {e}")

## 9. Olası Geliştirmeler

Bu proje temel haliyle birçok ihtiyacı karşılar, ancak şu geliştirmeler eklenerek daha kapsamlı hale getirilebilir:

| Geliştirme | Açıklama |
|------------|----------|
| **Sürücü Güncelleme Kontrolü** | `Win32_PnPSignedDriver` sınıfı ile yüklü sürücü versiyonlarını çekmek ve üreticinin sitesindeki son sürümle karşılaştırmak |
| **Dışa Aktarma (Export)** | JSON'a ek olarak Excel (`.xlsx`) ve PDF formatında rapor üretimi |
| **Threading ile Akıcı UI** | WMI sorgusu arka plan kanalında çalıştırılarak arayüzün başlangıçta donmaması |
| **Detay Penceresi** | Satıra tıklandığında açılan ayrı bir pencerede cihazın tüm WMI özelliklerini listeleme |
| **Sürücü Yedekleme** | `dism /online /export-driver` komutuyla mevcut sürücülerin yedeklenmesi |